# Gmail Source Demo

This notebook establishes a real Gmail read-only connection using credentials from the local `.env` file.

Required `.env` keys:
- `MAILMIND_GMAIL_CLIENT_ID`
- `MAILMIND_GMAIL_CLIENT_SECRET`

The notebook writes the OAuth token to `data/gmail_token.json` after first authorization.


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate repository root.")


def load_dotenv(dotenv_path: Path) -> None:
    if not dotenv_path.exists():
        return
    for raw_line in dotenv_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip("'").strip('"'))


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

load_dotenv(repo_root / ".env")
print(f"Using repo root: {repo_root}")


Using repo root: /Users/saketm10/Projects/openclaw_agents


In [2]:
required = {
    "MAILMIND_GMAIL_CLIENT_ID": os.getenv("MAILMIND_GMAIL_CLIENT_ID"),
    "MAILMIND_GMAIL_CLIENT_SECRET": os.getenv("MAILMIND_GMAIL_CLIENT_SECRET"),
}
missing = [key for key, value in required.items() if not value]
if missing:
    raise RuntimeError(f"Missing required .env keys: {missing}")

print("Found Gmail OAuth client credentials in .env")


Found Gmail OAuth client credentials in .env


In [3]:
from src.sources.gmail import GmailEmailSource


token_path = repo_root / "data" / "gmail_token.json"
source = GmailEmailSource(
    client_id=os.environ["MAILMIND_GMAIL_CLIENT_ID"],
    client_secret=os.environ["MAILMIND_GMAIL_CLIENT_SECRET"],
    token_path=token_path,
    max_results=5,
)

messages = source.fetch_new_messages()
print(f"Fetched {len(messages)} Gmail messages")
for message in messages[:5]:
    print({
        "source_id": message.source_id,
        "from_email": message.from_email,
        "subject": message.subject,
        "received_at": message.received_at.isoformat(),
    })


Fetched 5 Gmail messages
{'source_id': '19d57d4252fd25c3', 'from_email': 'noreply@glassdoor.com', 'subject': 'Hi all\n\nI’ve been actively exploring opportunities through Naukri and LinkedIn...', 'received_at': '2026-04-04T09:30:13+00:00'}
{'source_id': '19d57cbbd1cb8d73', 'from_email': 'noreply@swiggy.in', 'subject': '1.3 crore votes don’t lie! 👀', 'received_at': '2026-04-04T09:20:33+00:00'}
{'source_id': '19d57c71d0058c8b', 'from_email': 'no-reply@rmp.flipkart.com', 'subject': '₹0 Joining fee + ₹1,000 Voucher with Flipkart Axis Bank Credit card!', 'received_at': '2026-04-04T09:08:55+00:00'}
{'source_id': '19d57c4d1295ccea', 'from_email': 'hello@mails.pepperfry.com', 'subject': 'Invest in Everyday Living Sale Ending Tomorrow🪴', 'received_at': '2026-04-04T09:13:28+00:00'}
{'source_id': '19d57bd04ed7e442', 'from_email': 'alerts@hdfcbank.bank.in', 'subject': '❗  You have done a UPI txn. Check details!', 'received_at': '2026-04-04T09:04:57+00:00'}
